# ZemaVoice Stage 9 — Kids Voice Studio (personal experiment)
Test two youthful cartoon-style Amharic voice effects. **Not real child voices**: both use the same underlying adult single-speaker model. Pitch shifting may introduce artifacts. For noncommercial evaluation only: `facebook/mms-tts-amh` is CC BY-NC 4.0. Do not deploy commercially.

Run the cells from top to bottom. A GPU (T4) helps but CPU also works.


In [ ]:
!pip -q install 'transformers>=4.44,<5' 'accelerate>=0.26' 'soundfile>=0.12' 'uroman>=1.3' 'librosa>=0.10' 'ipywidgets>=8'


In [ ]:
import torch, numpy as np, librosa, soundfile as sf
from IPython.display import display, Audio
from transformers import VitsModel, VitsTokenizer, set_seed
from uroman import Uroman
import ipywidgets as widgets
from pathlib import Path
model_id='facebook/mms-tts-amh'
tokenizer=VitsTokenizer.from_pretrained(model_id)
model=VitsModel.from_pretrained(model_id)
romanizer=Uroman()
device='cuda' if torch.cuda.is_available() else 'cpu'
model=model.to(device).eval()
sr=int(model.config.sampling_rate)
print('Ready on',device,'| Sample rate:',sr)


In [ ]:
voice=widgets.Dropdown(options=['Kiki — cheerful cartoon girl (3-style)','Beni — playful cartoon boy (4-style)'],description='Voice:')
text_input=widgets.Textarea(value='ሰላም ጓደኞቼ! እንዴት ናችሁ? ዛሬ አብረን እንጫወታለን!',description='Amharic:',layout=widgets.Layout(width='95%',height='130px'))
show=widgets.VBox([voice,text_input]);display(show)
print('Edit the text and choose a voice, then run the next cell.')


In [ ]:
raw=text_input.value.strip()
assert raw,'Enter Amharic text first.'
assert len(raw)<=1000,'For the first test, use 1000 characters or fewer.'
romanized=romanizer.romanize_string(raw)
assert romanized,'Could not romanize this text.'
inputs=tokenizer(romanized,return_tensors='pt').to(device)
set_seed(42)
with torch.no_grad():
    audio=model(**inputs).waveform[0].detach().cpu().float().numpy()
# Experimental cartoon presets; pitch changes do NOT create genuine separate speakers.
chosen='kiki' if voice.value.startswith('Kiki') else 'beni'
semitones=3.0 if chosen=='kiki' else 1.5
processed=librosa.effects.pitch_shift(audio.astype(np.float32),sr=sr,n_steps=semitones)
processed=np.nan_to_num(processed)
processed=np.clip(processed,-0.98,0.98)
out=f'/content/zemavoice_stage9_{chosen}_cartoon.wav'
sf.write(out,processed,sr)
print('Saved:',out,'| Length:',round(len(processed)/sr,2),'seconds')
display(Audio(processed,rate=sr))


In [ ]:
from google.colab import files
files.download(out)
